In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Access the secret
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Log in to Hugging Face
login(token=hf_token)

In [ ]:
import pandas as pd

# The new file path copied from the Kaggle input pane
file_path = '/kaggle/input/apartments-for-rent-classified-10k/apartments_for_rent_classified_10K.csv' # Example path, yours might differ slightly

try:
  df = pd.read_csv(file_path, sep=';', encoding='cp1252', on_bad_lines='skip')
  print("Dataset loaded successfully! Here are the first few rows:")
  display(df.head())
except FileNotFoundError:
  print("File not found! Please double-check your file_path.")

In [ ]:
columns_to_keep = ['category', 'title', 'body', 'amenities', 'source']

df_text = df[columns_to_keep].copy()

print("DataFrame successfully scoped! Here's the new view:")
display(df_text.head())

print("\nAnd here's the info for our new DataFrame:")
df_text.info()

In [ ]:
df_text['amenities'].fillna('', inplace=True)

print("Missing values in 'amenities' handled. New info:")
df_text.info()

In [ ]:
# Install bitsandbytes for quantization and accelerate for model loading
!pip install -q -U bitsandbytes
!pip install -q -U accelerate

In [ ]:
import torch
from transformers import pipeline
import json

In [ ]:
# Initialize the pipeline with 4-bit quantization
text_gen_pipeline = pipeline(
    task="text-generation",
    model="google/gemma-3-4b-it",
    model_kwargs={
        "torch_dtype": torch.bfloat16,
        "quantization_config": {"load_in_4bit": True}
    },
)

print("Pipeline with quantized model loaded successfully!")

In [ ]:
def extract_key_phrases(ad_text):
    """
    Uses a Gemma model to extract a list of key phrases from a rental ad.
    Returns a list of strings or an empty list if extraction fails.
    """
    if not isinstance(ad_text, str) or not ad_text.strip():
        return []

    # The messages payload for the model
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant that extracts structured key phrases from rental advertisements for text analysis and always responds in valid JSON format.",
        },
        {
            "role": "user",
            "content": f"""
You are given a piece of text from a rental advertisement. Your task is to split the text into meaningful key phrases that capture important rental information.

Rules:
1. Keep related information together (e.g., full addresses, price ranges, unit descriptions).
2. Focus on phraces connected with useful details.some of the examples for such details are Location, Rental price, Unit type, Availability, Amenities etc.
3. Remove filler text, duplicates, or irrelevant words.
4. Return the output as valid JSON in the format: {{"key_phrases": ["phrase 1", "phrase 2"]}}

Text: "{ad_text}"
"""
        },
    ]

    try:
        # Generate the response
        response = text_gen_pipeline(messages, max_new_tokens=1000)

        # The model output is nested inside the response structure
        # The actual response starts after the user's message ends. We find the last message and get the content after it.
        generated_text = response[0]["generated_text"][-1]['content']

        # Clean up potential markdown formatting
        if generated_text.startswith("```json"):
            generated_text = generated_text[len("```json\n"):]
        if generated_text.endswith("```"):
            generated_text = generated_text[:-len("\n```")]

        # Parse the JSON string into a Python dictionary
        data = json.loads(generated_text)

        # Return the list of phrases
        return data.get("key_phrases", [])

    except (json.JSONDecodeError, IndexError, TypeError, KeyError) as e:
        print(f"⚠️ Error processing ad: {e}")
        print(f"   Problematic model output: {generated_text[:100]}...") # Print first 100 chars of problematic output
        return [] # Return an empty list on failure

In [ ]:
# Install tqdm for the progress bar
!pip install tqdm

In [ ]:
from tqdm import tqdm
import pandas as pd
import time
import os

# --- Configuration ---
batch_size = 100
# Path to the output from a PREVIOUS committed version
# Make sure to update 'your-previous-notebook-output' with the correct folder name
resume_file_path = '/kaggle/input/previous-notebook-output/dataset_with_key_phrases.csv' 

# Path for the NEW output of THIS session
output_filename = '/kaggle/working/dataset_with_key_phrases.csv'

# --- Resume Logic ---
processed_rows_list = []
start_index = 0

# Check if a file from a previous version exists
if os.path.exists(resume_file_path):
    print(f"Resuming from previous version's output file: {resume_file_path}")
    df_processed = pd.read_csv(resume_file_path)
    start_index = len(df_processed)
    processed_rows_list.append(df_processed)
    print(f"{start_index} rows already processed.")
else:
    print("Starting a new processing job.")
    
# Slice the main dataframe to only include the remaining, unprocessed rows
df_remaining = df_text.iloc[start_index:]

if df_remaining.empty:
    print("Processing is already complete!")
else:
    # Set tqdm to work well with pandas
    tqdm.pandas()

    print(f"Starting to process {len(df_remaining)} remaining rows in batches of {batch_size}...")

    # Create a list of batches to process from the remaining data
    list_df = [df_remaining[i:i+batch_size] for i in range(0, df_remaining.shape[0], batch_size)]

    # --- Main Processing Loop ---
    for i, batch_df in enumerate(tqdm(list_df, desc="Overall Progress")):
        
        # Apply our function to the 'body' column of the current batch
        batch_results = batch_df['body'].progress_apply(extract_key_phrases)
        
        # Assign the results to a new 'key_phrases' column in the batch
        batch_df['key_phrases'] = batch_results
        
        # Append the newly processed batch to our list of results
        processed_rows_list.append(batch_df)
        
        # --- Save Progress (Checkpoint) ---
        # After each batch, combine all processed rows so far and save to CSV
        final_df = pd.concat(processed_rows_list, ignore_index=True)
        final_df.to_csv(output_filename, index=False)
        
        time.sleep(2)

    print(f"\nProcessing complete! Your new dataset is saved at: {output_filename}")